### Medical Appointments No Show - Random Forest

1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix

2. Load the dataset and check for the missing values

In [ ]:
# Load the dataset you downloaded from Kaggle
df = pd.read_csv("dataset.csv")

# Display first few rows to understand structure
df.head()

3. CHECK & DROP MISSING VALUES

In [ ]:
# Check missing values in each column
print(df.isna().sum())

# Drop rows with ANY missing values (one line)
df = df.dropna()

# Confirm no missing values remain
print(df.isna().sum())

4. FEATURE EXTRACTION

In [ ]:
# Features to use for prediction (excluding identifiers and date columns)
feature_cols = [
    "Gender",
    "Age",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received"
]

# Target variable (Yes = No-show, No = Show)
target_col = "No-show"

# Create feature matrix X and target vector y
X = df[feature_cols]
y = df[target_col]

5. PREPROCESSING

In [ ]:
# Identify numeric and categorical features
numeric_features = ["Age"]
categorical_features = [
    "Gender",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received"
]

# Numeric preprocessing: Standard scaling
numeric_transformer = StandardScaler()

# Categorical preprocessing: One-hot encoding
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combine both preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

6. SPLIT THE DATA


In [ ]:
# First split: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: 10% validation, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

7. DECISION TREE — HYPERPARAMETER TUNING

In [ ]:
criteria = ["gini", "entropy", "log_loss"]

best_criterion = None
best_dt_val_accuracy = 0
best_dt_model = None

for crit in criteria:
    # Build pipeline: preprocessing + Decision Tree
    dt_pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("dt", DecisionTreeClassifier(criterion=crit, random_state=42))
    ])
    
    # Train model
    dt_pipeline.fit(X_train, y_train)
    
    # Predict on validation set
    y_val_pred = dt_pipeline.predict(X_val)
    
    # Compute accuracy
    val_acc = accuracy_score(y_val, y_val_pred)
    
    print(f"Decision Tree | criterion = {crit} | Validation Accuracy = {val_acc:.4f}")
    
    # Track best model
    if val_acc > best_dt_val_accuracy:
        best_dt_val_accuracy = val_acc
        best_criterion = crit
        best_dt_model = dt_pipeline

print("\nBest Decision Tree Criterion:", best_criterion)
print("Best Validation Accuracy:", best_dt_val_accuracy)

8. DECISION TREE — FINAL TEST EVALUATION


In [ ]:
# Predict on test set
y_test_pred_dt = best_dt_model.predict(X_test)

# Compute accuracy
dt_test_accuracy = accuracy_score(y_test, y_test_pred_dt)
print("\nFinal Decision Tree Test Accuracy:", dt_test_accuracy)

# Confusion matrix
dt_cm = confusion_matrix(y_test, y_test_pred_dt)
print("\nDecision Tree Confusion Matrix:\n", dt_cm)

9. RANDOM FOREST — HYPERPARAMETER TUNING

In [ ]:
n_estimators_list = [10, 50, 100, 200]

best_rf_n_estimators = None
best_rf_val_accuracy = 0
best_rf_model = None

for n in n_estimators_list:
    # Build pipeline: preprocessing + Random Forest
    rf_pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("rf", RandomForestClassifier(
            n_estimators=n,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Train model
    rf_pipeline.fit(X_train, y_train)
    
    # Predict on validation set
    y_val_pred_rf = rf_pipeline.predict(X_val)
    
    # Compute accuracy
    val_acc_rf = accuracy_score(y_val, y_val_pred_rf)
    
    print(f"Random Forest | n_estimators = {n} | Validation Accuracy = {val_acc_rf:.4f}")
    
    # Track best model
    if val_acc_rf > best_rf_val_accuracy:
        best_rf_val_accuracy = val_acc_rf
        best_rf_n_estimators = n
        best_rf_model = rf_pipeline

print("\nBest Random Forest n_estimators:", best_rf_n_estimators)
print("Best Validation Accuracy:", best_rf_val_accuracy)

10. RANDOM FOREST — FINAL TEST EVALUATION

In [ ]:
# Predict on test set
y_test_pred_rf = best_rf_model.predict(X_test)

# Compute accuracy
rf_test_accuracy = accuracy_score(y_test, y_test_pred_rf)
print("\nFinal Random Forest Test Accuracy:", rf_test_accuracy)

# Confusion matrix
rf_cm = confusion_matrix(y_test, y_test_pred_rf)
print("\nRandom Forest Confusion Matrix:\n", rf_cm)